# Etapa 7. Relação entre o SPI e o ENOS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/07_spi_x_enos.ipynb)

Este é o centro do trabalho. A pergunta: **o El Niño deixa o Paraná mais chuvoso e a La Niña o deixa mais seco? Em
que época do ano, com quanto atraso e com que intensidade? Igual em todo o estado?**

A resposta vem de duas formas que se completam:

- **Parte A, visual** (estações em destaque): séries com as fases, dispersão por estação do ano e mapa de calor da
  correlação por mês e defasagem;
- **Parte B, numérica** (todas as estações): correlação por estação do ano com **bootstrap em blocos**,
  correlação defasada, **tabela de contingência com probabilidade condicional e risco relativo**, Kruskal-Wallis
  e Dunn, e mapas.

Este notebook usa os arquivos gravados no Drive pelos notebooks 05 (`spi_pr.csv`) e 06 (`enos_mensal.csv`).
O capítulo 7 do caderno explica, com números simples, a autocorrelação e a tabela de contingência.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# Esta célula:
#   1) define DE ONDE os dados serão lidos (direto do GitHub, no Colab);
#   2) baixa as funções do projeto (pasta src/ do GitHub): fdp.py, spi.py, enos.py, estatistica.py;
#   3) conecta o seu Google Drive e usa a pasta spi_oni_pr/resultados, onde ficam os resultados
#      de uma etapa para a outra.
import sys
import urllib.request
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
REPO = "https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/"

if Path("../dados").exists():                   # computador do orientador, dentro da pasta do projeto
    DADOS = "../dados/"
    sys.path.append("../src")
else:                                           # Colab
    DADOS = REPO + "dados/"
    Path("src").mkdir(exist_ok=True)
    for arquivo in ["fdp.py", "spi.py", "enos.py", "estatistica.py"]:
        urllib.request.urlretrieve(REPO + "src/" + arquivo, "src/" + arquivo)   # baixa o arquivo
    sys.path.append("src")

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAIDA = Path("/content/drive/MyDrive/spi_oni_pr/resultados")
else:
    SAIDA = Path("../resultados")
SAIDA.mkdir(parents=True, exist_ok=True)
print("Dados lidos de:", DADOS)
print("Resultados gravados em:", SAIDA)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

est = pd.read_csv(DADOS + "estacoes_pr.csv", dtype={"cod": str, "cod_ibge": str})
mensal = pd.read_csv(DADOS + "chuva_mensal_pr.csv", dtype={"cod": str})
meso = gpd.read_file(DADOS + "pr_mesorregioes.geojson")
destaque = est[est["destaque"].notna()]           # as 4 estações em destaque
print(len(est), "estações;", len(mensal), "linhas de chuva mensal")

# Aparência padrão dos gráficos do trabalho (rode uma vez; vale para todos os gráficos do notebook)
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,       # tira a borda de cima
    "axes.spines.right": False,     # tira a borda da direita
    "axes.grid": True,              # grade de fundo...
    "grid.color": "#e6e5e0",        # ...bem clara, para não competir com os dados
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,         # grade atrás das barras e linhas
})

AZUL = "#2a78d6"                    # cor única quando há só uma série
# uma cor fixa para cada estação em destaque (a cor acompanha a estação em todas as figuras)
CORES = {"Santa Helena": "#2a78d6", "Cascavel": "#eb6834", "Foz do Iguaçu": "#1baf7a", "Mercedes": "#eda100"}

from matplotlib.colors import LinearSegmentedColormap

# escala divergente: vermelho (seco / negativo) - cinza (zero) - azul (úmido / positivo)
DIVERGENTE = LinearSegmentedColormap.from_list("div", ["#a3201f", "#e34948", "#f0efec", "#2a78d6", "#184f95"])
# cores das fases do ENOS: El Niño = águas quentes (vermelho), La Niña = águas frias (azul)
CORES_FASE = {"El Niño": "#e34948", "Neutro": "#9a9890", "La Niña": "#2a78d6"}

def mapa_base(ax):
    """Desenha os limites das mesorregiões no fundo de um mapa."""
    meso.plot(ax=ax, color="#f4f3ef", edgecolor="#9a9890", linewidth=0.6)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for lado in ["left", "bottom"]:
        ax.spines[lado].set_visible(False)

import estatistica as E
import spi

spi_pr = pd.read_csv(SAIDA / "spi_pr.csv", dtype={"cod": str})
enos_m = pd.read_csv(SAIDA / "enos_mensal.csv")
print(len(spi_pr), "linhas de SPI;", len(enos_m), "meses de ONI")

## Preparação: os pares por estação do ano

O SPI-3 de um mês é a chuva dele e dos dois meses anteriores. Assim, o SPI-3 de **fevereiro** é a chuva de
dezembro, janeiro e fevereiro (o verão), o mesmo trimestre do ONI **DJF** (que está no mês central, janeiro).
O dicionário abaixo diz, para cada estação do ano, em que mês está o ONI e em que mês está o SPI-3 do mesmo
trimestre.

In [ ]:
ESTACOES_ANO = {                  # estação do ano: (mês do ONI, mês do SPI-3)
    "Verão (DJF)": (1, 2),
    "Outono (MAM)": (4, 5),
    "Inverno (JJA)": (7, 8),
    "Primavera (SON)": (10, 11),
}

partes = []
for nome, (mes_oni, mes_spi) in ESTACOES_ANO.items():
    a = spi_pr[spi_pr["mes"] == mes_spi][["cod", "ano", "spi3"]]
    b = enos_m[enos_m["mes"] == mes_oni][["ano", "oni", "fase", "intensidade"]]
    junto = a.merge(b, on="ano")                # junta pelo ano
    junto["estacao_ano"] = nome
    partes.append(junto)
sazonal = pd.concat(partes).dropna(subset=["spi3"])
sazonal["classe"] = sazonal["spi3"].apply(spi.classe3)        # seco / normal / úmido
sazonal.head()

---

# Parte A. Visual

## A1. Séries do SPI-12 com as fases do ENOS (Figura 8 do TCC)

Faixas vermelhas = El Niño; azuis = La Niña. Repare se os períodos de SPI-12 positivo coincidem com as faixas
vermelhas.

In [ ]:
enos_m["data"] = pd.to_datetime(dict(year=enos_m["ano"], month=enos_m["mes"], day=1))

fig, eixos = plt.subplots(4, 1, figsize=(12, 9), sharex=True, sharey=True)
for ax, (i, linha) in zip(eixos, destaque.iterrows()):
    # faixas das fases: um retângulo para cada mês de El Niño ou La Niña
    for j, mes in enos_m.iterrows():
        if mes["fase"] != "Neutro":
            ax.axvspan(mes["data"], mes["data"] + pd.DateOffset(months=1), color=CORES_FASE[mes["fase"]],
                       alpha=0.18, linewidth=0)
    tab = spi_pr[spi_pr["cod"] == linha["cod"]].dropna(subset=["spi12"])
    datas = pd.to_datetime(dict(year=tab["ano"], month=tab["mes"], day=1))
    ax.plot(datas, tab["spi12"], color="black", linewidth=1.2)
    ax.axhline(0, color="#9a9890", linewidth=0.8)
    ax.set_title(linha["municipio"], fontsize=9, loc="left")
    ax.set_ylabel("SPI-12")
fig.suptitle("SPI-12 e fases do ENOS (vermelho: El Niño; azul: La Niña)")
fig.tight_layout()
fig.savefig(SAIDA / "fig08_spi12_fases_enos.png", dpi=300, bbox_inches="tight")
plt.show()

## A2. Dispersão ONI × SPI-3 por estação do ano (Figura 9 do TCC)

Santa Helena. Cada ponto é um ano; a cor é a fase do ENOS naquele trimestre.

In [ ]:
COD_EXEMPLO = "2454012"          # Santa Helena (troque para ver outra estação)

fig, eixos = plt.subplots(1, 4, figsize=(15, 3.8), sharex=True, sharey=True)
for ax, nome in zip(eixos, ESTACOES_ANO):
    tab = sazonal[(sazonal["cod"] == COD_EXEMPLO) & (sazonal["estacao_ano"] == nome)]
    for fase, cor in CORES_FASE.items():
        parte = tab[tab["fase"] == fase]
        ax.scatter(parte["oni"], parte["spi3"], color=cor, s=26, label=fase, edgecolor="white", linewidth=0.5)
    r = E.correlacao(tab["oni"], tab["spi3"])
    ax.set_title(f"{nome}  (r = {r:.2f})", fontsize=9)
    ax.axhline(0, color="#9a9890", linewidth=0.8)
    ax.axvline(0, color="#9a9890", linewidth=0.8)
    ax.set_xlabel("ONI (°C)")
eixos[0].set_ylabel("SPI-3")
eixos[0].legend(fontsize=8)
nome_est = est.loc[est["cod"] == COD_EXEMPLO, "municipio"].iloc[0]
fig.suptitle(f"ONI × SPI-3 por estação do ano, {nome_est}")
fig.tight_layout()
fig.savefig(SAIDA / "fig09_dispersao_oni_spi3.png", dpi=300, bbox_inches="tight")
plt.show()

## A3. Correlação por mês e defasagem (Figura 10 do TCC)

Para cada mês do SPI-3 (linhas) e cada **defasagem** de 0 a 12 meses (colunas): a correlação entre o SPI-3 daquele
mês e o ONI de `defasagem` meses antes, ao longo dos anos. Azul = correlação positiva (El Niño → mais chuva).
A célula mais azul diz **em que época** e **com quanto atraso** o sinal do Pacífico aparece na chuva.

In [ ]:
oni_serie = enos_m.set_index("data")["oni"]                     # ONI com a data como índice


def mapa_de_calor(cod):
    tab = spi_pr[spi_pr["cod"] == cod].copy()
    tab["data"] = pd.to_datetime(dict(year=tab["ano"], month=tab["mes"], day=1))
    matriz = np.full((12, 13), np.nan)                          # 12 meses × 13 defasagens
    for m in range(1, 13):
        do_mes = tab[tab["mes"] == m]
        for defasagem in range(13):
            datas_oni = do_mes["data"] - pd.DateOffset(months=defasagem)   # o ONI de "defasagem" meses antes
            oni_antes = oni_serie.reindex(datas_oni).values
            matriz[m - 1, defasagem] = E.correlacao(oni_antes, do_mes["spi3"].values)
    return matriz


matriz = mapa_de_calor(COD_EXEMPLO)
fig, ax = plt.subplots(figsize=(9, 5))
img = ax.imshow(matriz, cmap=DIVERGENTE, vmin=-0.6, vmax=0.6, aspect="auto")
for m in range(12):                                            # escreve o valor em cada célula
    for d in range(13):
        ax.text(d, m, f"{matriz[m, d]:.2f}", ha="center", va="center", fontsize=6)
ax.set_xticks(range(13))
ax.set_xlabel("Defasagem do ONI (meses antes do SPI-3)")
ax.set_yticks(range(12))
ax.set_yticklabels(["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"])
ax.set_ylabel("Mês do SPI-3")
ax.grid(False)
fig.colorbar(img, ax=ax, label="Correlação de Pearson")
ax.set_title(f"Correlação ONI × SPI-3, {nome_est}")
fig.savefig(SAIDA / "fig10_correlacao_mes_defasagem.png", dpi=300, bbox_inches="tight")
plt.show()

---

# Parte B. Numérica

## B4. Correlação por estação do ano, com bootstrap em blocos (Tabela 7 do TCC)

Para cada estação pluviométrica e cada estação do ano: correlação de Pearson e de Spearman entre o ONI e o SPI-3
(um par por ano) e o intervalo de 95% por **bootstrap em blocos de 3 anos** (500 reamostras). A correlação é
significativa se o intervalo não contém o zero. Leva cerca de um minuto.

In [ ]:
linhas = []
for cod in est["cod"]:
    for nome in ESTACOES_ANO:
        tab = sazonal[(sazonal["cod"] == cod) & (sazonal["estacao_ano"] == nome)].sort_values("ano")
        r, baixo, alto, significativa = E.bootstrap_blocos(tab["oni"], tab["spi3"], bloco=3, n_reamostras=500)
        linhas.append({"cod": cod, "estacao_ano": nome, "r_pearson": r,
                       "r_spearman": E.correlacao(tab["oni"], tab["spi3"], "spearman"),
                       "ic_inferior": baixo, "ic_superior": alto, "significativa": significativa})
cor_sazonal = pd.DataFrame(linhas).merge(est[["cod", "municipio", "mesorregiao", "lat", "lon"]], on="cod")
cor_sazonal.to_csv(SAIDA / "correlacao_sazonal.csv", index=False)

tabela7 = cor_sazonal.groupby("estacao_ano").agg(
    r_medio=("r_pearson", "mean"), r_minimo=("r_pearson", "min"), r_maximo=("r_pearson", "max"),
    rho_medio=("r_spearman", "mean"), pct_significativas=("significativa", "mean"))
tabela7["pct_significativas"] = tabela7["pct_significativas"] * 100
tabela7 = tabela7.reindex(list(ESTACOES_ANO)).round(2)
tabela7.to_csv(SAIDA / "tabela07_correlacao_sazonal.csv")
tabela7

A mesma correlação, na primavera, por mesorregião:

In [ ]:
primavera = cor_sazonal[cor_sazonal["estacao_ano"] == "Primavera (SON)"]
primavera.groupby("mesorregiao")[["r_pearson", "significativa"]].mean().sort_values("r_pearson").round(2)

## B5. Correlação defasada do SPI-12

O SPI-12 responde ao acumulado do ano, então o efeito do ENOS pode aparecer com atraso. Para cada estação, a
correlação entre o SPI-12 e o ONI de 0 a 12 meses antes, e a defasagem de maior correlação.

In [ ]:
spi_pr["data"] = pd.to_datetime(dict(year=spi_pr["ano"], month=spi_pr["mes"], day=1))
linhas = []
for cod in est["cod"]:
    tab = spi_pr[spi_pr["cod"] == cod].dropna(subset=["spi12"])
    for defasagem in range(13):
        oni_antes = oni_serie.reindex(tab["data"] - pd.DateOffset(months=defasagem)).values
        linhas.append({"cod": cod, "defasagem": defasagem, "r": E.correlacao(oni_antes, tab["spi12"].values)})
defasada = pd.DataFrame(linhas)

maior = defasada.loc[defasada.groupby("cod")["r"].idxmax()]
print("Defasagem de maior correlação (meses) - quantas estações em cada uma:")
print(maior["defasagem"].value_counts().sort_index())
print("Correlação máxima média:", round(maior["r"].mean(), 2))

Nas estações em destaque, a significância da correlação na melhor defasagem, por bootstrap em blocos de 24 meses
(o SPI-12 tem forte dependência entre meses vizinhos).

In [ ]:
for i, linha in destaque.iterrows():
    tab = spi_pr[spi_pr["cod"] == linha["cod"]].dropna(subset=["spi12"])
    d = int(maior.loc[maior["cod"] == linha["cod"], "defasagem"].iloc[0])
    oni_antes = oni_serie.reindex(tab["data"] - pd.DateOffset(months=d)).values
    r, baixo, alto, sig = E.bootstrap_blocos(oni_antes, tab["spi12"].values, bloco=24, n_reamostras=300)
    print(f"{linha['municipio']:15s} defasagem {d:2d} meses: r = {r:.2f}  IC95% [{baixo:.2f}; {alto:.2f}]  "
          f"{'significativa' if sig else 'não significativa'}")

## B6. Tabela de contingência: probabilidade condicional e risco relativo (Tabela 8 do TCC)

Esta é a análise que mais interessa ao produtor e ao seguro agrícola: **qual a chance de uma primavera seca num ano
de La Niña, comparada com a chance num ano qualquer?**

- **Probabilidade condicional** P(seco | La Niña): dos anos de La Niña, em quantos % a primavera foi seca
  (SPI-3 ≤ −1)?
- **Probabilidade climatológica** P(seco): em quantos % de todos os anos a primavera foi seca?
- **Risco relativo** = P(seco | La Niña) ÷ P(seco). RR = 2 quer dizer "o dobro da chance".

Primeiro, nas estações em destaque, a tabela completa:

In [ ]:
for i, linha in destaque.iterrows():
    tab = sazonal[(sazonal["cod"] == linha["cod"]) & (sazonal["estacao_ano"] == "Primavera (SON)")]
    res = E.contingencia(tab["fase"], tab["classe"])
    print("=====", linha["municipio"], "| qui-quadrado: p =", round(res["p_valor"], 3))
    print("Número de anos:")
    print(res["tabela"])
    print("P(classe | fase), %:")
    print(res["prob"])
    print("Risco relativo:")
    print(res["risco"])
    print()

Agora para todas as estações e as quatro estações do ano: P(seco | La Niña), P(úmido | El Niño) e os riscos
relativos.

In [ ]:
linhas = []
for cod in est["cod"]:
    for nome in ESTACOES_ANO:
        tab = sazonal[(sazonal["cod"] == cod) & (sazonal["estacao_ano"] == nome)]
        res = E.contingencia(tab["fase"], tab["classe"])
        prob, clima = res["prob"], res["clima"]
        linhas.append({
            "cod": cod, "estacao_ano": nome,
            "p_seco_lanina": prob.loc["La Niña", "seco"] if "seco" in prob else 0.0,
            "p_seco_clima": clima.get("seco", 0.0),
            "p_umido_elnino": prob.loc["El Niño", "úmido"] if "úmido" in prob else 0.0,
            "p_umido_clima": clima.get("úmido", 0.0),
            "p_quiquadrado": res["p_valor"]})
probs = pd.DataFrame(linhas).merge(est[["cod", "municipio", "mesorregiao", "lat", "lon"]], on="cod")
probs["rr_seco_lanina"] = probs["p_seco_lanina"] / probs["p_seco_clima"]
probs["rr_umido_elnino"] = probs["p_umido_elnino"] / probs["p_umido_clima"]
probs.to_csv(SAIDA / "probabilidade_condicional.csv", index=False)

tabela8 = probs.groupby("estacao_ano")[["p_seco_lanina", "p_seco_clima", "rr_seco_lanina",
                                         "p_umido_elnino", "p_umido_clima", "rr_umido_elnino"]].mean()
tabela8 = tabela8.reindex(list(ESTACOES_ANO)).round(2)
tabela8.to_csv(SAIDA / "tabela08_probabilidade_condicional.csv")
tabela8

## B7. Kruskal-Wallis e Dunn: há efeito "dose-resposta"?

O SPI-3 da primavera difere entre as fases? E entre as intensidades do El Niño? Os grupos: La Niña, Neutro,
El Niño fraco ou moderado e El Niño forte ou muito forte. Se o efeito tiver "dose-resposta", o SPI sobe nessa ordem.

In [ ]:
def grupo(linha):
    if linha["fase"] == "El Niño":
        if linha["intensidade"] in ["forte", "muito forte"]:
            return "El Niño forte"
        return "El Niño fraco/moderado"
    return linha["fase"]


sazonal["grupo"] = sazonal.apply(grupo, axis=1)       # apply aplica a função em cada linha

for i, linha in destaque.iterrows():
    tab = sazonal[(sazonal["cod"] == linha["cod"]) & (sazonal["estacao_ano"] == "Primavera (SON)")]
    p_kw, dunn = E.kruskal_dunn(tab["spi3"], tab["grupo"])
    print("=====", linha["municipio"], "| Kruskal-Wallis: p =", round(p_kw, 4))
    print("SPI-3 mediano por grupo:", tab.groupby("grupo")["spi3"].median().round(2).to_dict())
    print(dunn)
    print()

Em quantas estações o Kruskal-Wallis (fases El Niño, Neutro, La Niña) é significativo, em cada estação do ano?

In [ ]:
linhas = []
for cod in est["cod"]:
    for nome in ESTACOES_ANO:
        tab = sazonal[(sazonal["cod"] == cod) & (sazonal["estacao_ano"] == nome)]
        p_kw, _ = E.kruskal_dunn(tab["spi3"], tab["fase"])
        linhas.append({"cod": cod, "estacao_ano": nome, "p_kw": p_kw})
kw = pd.DataFrame(linhas)
kw["significativo"] = kw["p_kw"] < 0.05
(kw.groupby("estacao_ano")["significativo"].mean() * 100).reindex(list(ESTACOES_ANO)).round(0)

## B8. Mapas (Figura 11 do TCC) e resumo por mesorregião (Tabela 9)

Quatro mapas: a correlação na primavera e no inverno, P(seco | La Niña) e P(úmido | El Niño) na primavera. Nos dois
primeiros, as estações com correlação significativa têm contorno preto.

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(13, 9.5))

for ax, nome in zip(eixos[0], ["Primavera (SON)", "Inverno (JJA)"]):
    mapa_base(ax)
    tab = cor_sazonal[cor_sazonal["estacao_ano"] == nome]
    contorno = np.where(tab["significativa"], "black", "white")
    pontos = ax.scatter(tab["lon"], tab["lat"], c=tab["r_pearson"], cmap=DIVERGENTE, vmin=-0.6, vmax=0.6,
                        s=24, edgecolor=contorno, linewidth=0.5)
    ax.set_title(f"Correlação ONI × SPI-3, {nome}", fontsize=10)
    fig.colorbar(pontos, ax=ax, label="r de Pearson", shrink=0.7)

azuis = LinearSegmentedColormap.from_list("azuis", ["#cde2fb", "#6da7ec", "#2a78d6", "#184f95", "#0d366b"])
vermelhos = LinearSegmentedColormap.from_list("verm", ["#fbe3e2", "#f08a89", "#e34948", "#a3201f", "#6e1312"])
prim = probs[probs["estacao_ano"] == "Primavera (SON)"]
for ax, coluna, cmap, titulo in [(eixos[1, 0], "p_seco_lanina", vermelhos, "P(primavera seca | La Niña), %"),
                                 (eixos[1, 1], "p_umido_elnino", azuis, "P(primavera úmida | El Niño), %")]:
    mapa_base(ax)
    pontos = ax.scatter(prim["lon"], prim["lat"], c=prim[coluna], cmap=cmap, vmin=0, vmax=70, s=24,
                        edgecolor="white", linewidth=0.3)
    ax.set_title(titulo, fontsize=10)
    fig.colorbar(pontos, ax=ax, label="%", shrink=0.7)

fig.tight_layout()
fig.savefig(SAIDA / "fig11_mapas_spi_enos.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
tabela9 = (cor_sazonal[cor_sazonal["estacao_ano"] == "Primavera (SON)"].groupby("mesorregiao")
           .agg(estacoes=("cod", "count"), r_primavera=("r_pearson", "mean"),
                pct_signif=("significativa", "mean")))
tabela9["pct_signif"] = tabela9["pct_signif"] * 100
extra = prim.groupby("mesorregiao")[["p_seco_lanina", "rr_seco_lanina", "p_umido_elnino", "rr_umido_elnino"]].mean()
tabela9 = tabela9.join(extra).sort_values("r_primavera", ascending=False).round(2)
tabela9.to_csv(SAIDA / "tabela09_resumo_mesorregiao.csv")
tabela9

---

## O que observar

- A literatura indica sinal mais forte na **primavera** e mais intenso no **Sul e no Oeste** do que no Norte
  (Grimm; Barros; Doyle, 2000; De Bodas Terassi et al., 2018). Os mapas mostram isso?
- Compare P(seco | La Niña) com P(úmido | El Niño): a resposta é **simétrica**? Essa é a hipótese do trabalho.
- Uma correlação de 0,3 parece pequena, mas pode dobrar a chance de uma primavera seca. Por isso a probabilidade
  condicional é o número que vai para as conclusões.

## Pontos de decisão com o orientador

Tamanho do bloco do bootstrap (3 anos no sazonal, 24 meses no SPI-12) e quais defasagens entram no texto.

## Exercícios

**Exercício 1.** Refaça as figuras A2 e A3 para Cascavel (mude `COD_EXEMPLO`). O padrão é o mesmo de Santa Helena?

In [ ]:
# escreva sua resposta aqui

**Exercício 2.** Escreva, com os números da Tabela 8, a frase do item 5.6.2: "em anos de La Niña, a probabilidade
de primavera seca foi de ...%, contra ...% considerando todos os anos".

*Escreva sua resposta aqui.*

**Exercício 3.** No Kruskal-Wallis por grupos, o SPI-3 mediano sobe na ordem La Niña → Neutro → El Niño fraco →
El Niño forte? O que isso diz sobre a intensidade do evento?

*Escreva sua resposta aqui.*

---

**Entregar.** `fig08` a `fig11`, `tabela07` a `tabela09` e os exercícios. **O que escrever:** item 5.6 (5.6.1 a
5.6.3) do TCC2, o núcleo dos resultados.